# AgenticCodingLoop: verified state machine in Python

Generated by [nuxmv-editor-js](https://github.com/belkassaby/nuxmv-editor-js) from `agent-coding-loop.nxd`. The diagram was model checked with nuXmv; this notebook runs the **same** transition system as Python code:

* `send(event)` only follows transitions of the verified model, anything else raises `InvalidTransition`;
* `allowed_events()` gives the legal next steps, e.g. to constrain what an LLM may choose;
* runtime monitors re-check the properties that can be decided on a running system.

| Kind | Name | Property | nuXmv | Monitored at run time |
| --- | --- | --- | --- | --- |
| LTLSPEC | `tested_before_agent_review` | `G (reviewer = security_agent -> Y phase = testing)` | — | yes |
| LTLSPEC | `security_then_quality` | `G (reviewer = quality_agent -> Y reviewer = security_agent)` | — | yes |
| LTLSPEC | `agents_before_human` | `G (phase = human_review -> Y reviewer = quality_agent)` | — | yes |
| LTLSPEC | `human_merges` | `G (phase = merged -> Y (phase = human_review & actor = human))` | — | yes |
| INVARSPEC | `review_rounds_bounded` | `agent_review_round <= 2` | — | yes |
| INVARSPEC | `retry_budget` | `test_failures <= 2` | — | yes |
| LTLSPEC | `bounded_rework` | `G (phase = agent_review -> F (phase = human_review \| phase = prompting))` | — | no (future-time operator F) |
| LTLSPEC | `human_decides_release` | `G (phase = deploying -> Y (phase = release_decision & actor = human))` | — | yes |
| LTLSPEC | `deploys_the_merged_change` | `G (phase = deploying -> (phase != prompting S phase = merged))` | — | yes |
| LTLSPEC | `staging_passed_for_this_release` | `G (phase = deploying -> (phase != prompting S phase = smoke_testing))` | — | yes |
| LTLSPEC | `rollback_goes_to_human` | `G (phase = rolled_back -> X phase = release_decision)` | — | no (future-time operator X) |
| CTLSPEC | `can_go_live` | `AG EF phase = live` | — | no (CTL quantifies over all possible futures) |
| LTLSPEC | `always_goes_live` | `F phase = live` | — | no (not of the form G(present/past formula)) |

In [ ]:
# The live diagram widget needs anywidget (the rest has no dependency).
%pip install --quiet anywidget

In [ ]:
%%writefile agentic_coding_loop_fsm.py
"""AgenticCodingLoop: state machine generated by nuxmv-editor-js from agent-coding-loop.nxd.

The transition table below is the one verified with nuXmv: send() rejects any other move.
Properties checked by nuXmv on the diagram:
  - LTLSPEC tested_before_agent_review: G (reviewer = security_agent -> Y phase = testing)   [also monitored at run time]
  - LTLSPEC security_then_quality: G (reviewer = quality_agent -> Y reviewer = security_agent)   [also monitored at run time]
  - LTLSPEC agents_before_human: G (phase = human_review -> Y reviewer = quality_agent)   [also monitored at run time]
  - LTLSPEC human_merges: G (phase = merged -> Y (phase = human_review & actor = human))   [also monitored at run time]
  - INVARSPEC review_rounds_bounded: agent_review_round <= 2   [also monitored at run time]
  - INVARSPEC retry_budget: test_failures <= 2   [also monitored at run time]
  - LTLSPEC bounded_rework: G (phase = agent_review -> F (phase = human_review | phase = prompting))   [static only: future-time operator F]
  - LTLSPEC human_decides_release: G (phase = deploying -> Y (phase = release_decision & actor = human))   [also monitored at run time]
  - LTLSPEC deploys_the_merged_change: G (phase = deploying -> (phase != prompting S phase = merged))   [also monitored at run time]
  - LTLSPEC staging_passed_for_this_release: G (phase = deploying -> (phase != prompting S phase = smoke_testing))   [also monitored at run time]
  - LTLSPEC rollback_goes_to_human: G (phase = rolled_back -> X phase = release_decision)   [static only: future-time operator X]
  - CTLSPEC can_go_live: AG EF phase = live   [static only: CTL quantifies over all possible futures]
  - LTLSPEC always_goes_live: F phase = live   [static only: not of the form G(present/past formula)]

Usage:
    fsm = AgenticCodingLoopFSM()
    fsm.allowed_events()          # legal next events (e.g. to constrain an LLM)
    fsm.send("USER_SUBMIT")
    fsm.widget()                  # live diagram in Jupyter (pip install anywidget)
    fsm.link_editor(channel="me") # live state in nuxmv-editor
"""

from __future__ import annotations

import enum


class State(str, enum.Enum):
    PROMPTING = "prompting"  # human prompt
    DESIGNING = "designing"  # agent plans, human reviews
    DEVELOPING0 = "developing0"  # agent codes
    TESTING0 = "testing0"  # tests run
    DEVELOPING1 = "developing1"  # self-heal #1
    TESTING1 = "testing1"  # tests run #2
    DEVELOPING2 = "developing2"  # self-heal #2
    TESTING2 = "testing2"  # tests run #3
    SECURITY_REVIEW1 = "security_review1"  # security agent review
    QUALITY_REVIEW1 = "quality_review1"  # quality agent review
    REWORK_DEVELOPING = "rework_developing"  # agent reworks review findings
    REWORK_TESTING = "rework_testing"  # tests run after rework
    SECURITY_REVIEW2 = "security_review2"  # security agent re-review
    QUALITY_REVIEW2 = "quality_review2"  # quality agent re-review
    HUMAN_REVIEW = "human_review"  # human git diff review
    MERGED = "merged"  # PR merged
    BUILDING = "building"  # build release candidate
    STAGING = "staging"  # deploy to staging
    SMOKE_TESTING = "smoke_testing"  # staging smoke tests
    RELEASE_DECISION = "release_decision"  # human: release?
    ON_HOLD = "on_hold"  # release held
    DEPLOYING = "deploying"  # deploy to production
    LIVE = "live"  # live in production
    ROLLED_BACK = "rolled_back"  # rolled back


class Event(str, enum.Enum):
    USER_SUBMIT = "USER_SUBMIT"
    PLAN_ACCEPTED = "PLAN_ACCEPTED"
    HUMAN_CLARIFIES = "HUMAN_CLARIFIES"
    CODE_WRITTEN = "CODE_WRITTEN"
    TESTS_PASSED = "TESTS_PASSED"
    TESTS_FAILED = "TESTS_FAILED"
    MAX_RETRIES_REACHED = "MAX_RETRIES_REACHED"
    SECURITY_APPROVED = "SECURITY_APPROVED"
    CHANGES_REQUESTED = "CHANGES_REQUESTED"
    QUALITY_APPROVED = "QUALITY_APPROVED"
    TESTS_FAILED_ESCALATE = "TESTS_FAILED_ESCALATE"
    REJECTED_AGAIN_ESCALATE = "REJECTED_AGAIN_ESCALATE"
    HUMAN_APPROVED = "HUMAN_APPROVED"
    HUMAN_REJECTED = "HUMAN_REJECTED"
    REDESIGN = "REDESIGN"
    CI_BUILD = "CI_BUILD"
    BUILD_OK = "BUILD_OK"
    BUILD_FAILED = "BUILD_FAILED"
    DEPLOYED_TO_STAGING = "DEPLOYED_TO_STAGING"
    SMOKE_PASSED = "SMOKE_PASSED"
    SMOKE_FAILED = "SMOKE_FAILED"
    HUMAN_APPROVES_RELEASE = "HUMAN_APPROVES_RELEASE"
    HUMAN_HOLDS = "HUMAN_HOLDS"
    HUMAN_REJECTS_RELEASE = "HUMAN_REJECTS_RELEASE"
    HUMAN_REVISITS = "HUMAN_REVISITS"
    HEALTH_OK = "HEALTH_OK"
    HEALTH_CHECK_FAILED = "HEALTH_CHECK_FAILED"
    BACK_TO_HUMAN = "BACK_TO_HUMAN"


#: Initial states of the model.
INITIAL_STATES = (State.PROMPTING,)
#: States without outgoing transitions (they stutter in the nuXmv model).
TERMINAL_STATES = frozenset({State.LIVE})

#: (state, event) -> next state: exactly the transitions of the verified diagram.
TRANSITIONS = {
    (State.PROMPTING, Event.USER_SUBMIT): State.DESIGNING,  # USER_SUBMIT
    (State.DESIGNING, Event.PLAN_ACCEPTED): State.DEVELOPING0,  # PLAN_ACCEPTED
    (State.DESIGNING, Event.HUMAN_CLARIFIES): State.PROMPTING,  # human_clarifies
    (State.DEVELOPING0, Event.CODE_WRITTEN): State.TESTING0,  # CODE_WRITTEN
    (State.TESTING0, Event.TESTS_PASSED): State.SECURITY_REVIEW1,  # TESTS_PASSED
    (State.TESTING0, Event.TESTS_FAILED): State.DEVELOPING1,  # TESTS_FAILED
    (State.DEVELOPING1, Event.CODE_WRITTEN): State.TESTING1,  # CODE_WRITTEN
    (State.TESTING1, Event.TESTS_PASSED): State.SECURITY_REVIEW1,  # TESTS_PASSED
    (State.TESTING1, Event.TESTS_FAILED): State.DEVELOPING2,  # TESTS_FAILED
    (State.DEVELOPING2, Event.CODE_WRITTEN): State.TESTING2,  # CODE_WRITTEN
    (State.TESTING2, Event.TESTS_PASSED): State.SECURITY_REVIEW1,  # TESTS_PASSED
    (State.TESTING2, Event.MAX_RETRIES_REACHED): State.PROMPTING,  # max_retries_reached
    (State.SECURITY_REVIEW1, Event.SECURITY_APPROVED): State.QUALITY_REVIEW1,  # security_approved
    (State.SECURITY_REVIEW1, Event.CHANGES_REQUESTED): State.REWORK_DEVELOPING,  # changes_requested
    (State.QUALITY_REVIEW1, Event.QUALITY_APPROVED): State.HUMAN_REVIEW,  # quality_approved
    (State.QUALITY_REVIEW1, Event.CHANGES_REQUESTED): State.REWORK_DEVELOPING,  # changes_requested
    (State.REWORK_DEVELOPING, Event.CODE_WRITTEN): State.REWORK_TESTING,  # CODE_WRITTEN
    (State.REWORK_TESTING, Event.TESTS_PASSED): State.SECURITY_REVIEW2,  # TESTS_PASSED
    (State.REWORK_TESTING, Event.TESTS_FAILED_ESCALATE): State.PROMPTING,  # TESTS_FAILED: escalate
    (State.SECURITY_REVIEW2, Event.SECURITY_APPROVED): State.QUALITY_REVIEW2,  # security_approved
    (State.SECURITY_REVIEW2, Event.REJECTED_AGAIN_ESCALATE): State.PROMPTING,  # rejected again: escalate
    (State.QUALITY_REVIEW2, Event.QUALITY_APPROVED): State.HUMAN_REVIEW,  # quality_approved
    (State.QUALITY_REVIEW2, Event.REJECTED_AGAIN_ESCALATE): State.PROMPTING,  # rejected again: escalate
    (State.HUMAN_REVIEW, Event.HUMAN_APPROVED): State.MERGED,  # human_approved
    (State.HUMAN_REVIEW, Event.HUMAN_REJECTED): State.PROMPTING,  # human_rejected
    (State.HUMAN_REVIEW, Event.REDESIGN): State.DESIGNING,  # redesign
    (State.MERGED, Event.CI_BUILD): State.BUILDING,  # CI build
    (State.BUILDING, Event.BUILD_OK): State.STAGING,  # build_ok
    (State.BUILDING, Event.BUILD_FAILED): State.PROMPTING,  # build_failed
    (State.STAGING, Event.DEPLOYED_TO_STAGING): State.SMOKE_TESTING,  # deployed_to_staging
    (State.SMOKE_TESTING, Event.SMOKE_PASSED): State.RELEASE_DECISION,  # smoke_passed
    (State.SMOKE_TESTING, Event.SMOKE_FAILED): State.PROMPTING,  # smoke_failed
    (State.RELEASE_DECISION, Event.HUMAN_APPROVES_RELEASE): State.DEPLOYING,  # human_approves_release
    (State.RELEASE_DECISION, Event.HUMAN_HOLDS): State.ON_HOLD,  # human_holds
    (State.RELEASE_DECISION, Event.HUMAN_REJECTS_RELEASE): State.PROMPTING,  # human_rejects_release
    (State.ON_HOLD, Event.HUMAN_REVISITS): State.RELEASE_DECISION,  # human_revisits
    (State.DEPLOYING, Event.HEALTH_OK): State.LIVE,  # health_ok
    (State.DEPLOYING, Event.HEALTH_CHECK_FAILED): State.ROLLED_BACK,  # health_check_failed
    (State.ROLLED_BACK, Event.BACK_TO_HUMAN): State.RELEASE_DECISION,  # back to human
}

#: Labelling L(s): attribute values in each state (None = any value, set it with send(..., values=)).
LABELS = {
    State.PROMPTING: {"phase": "prompting", "actor": "human", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.DESIGNING: {"phase": "designing", "actor": "agent", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.DEVELOPING0: {"phase": "developing", "actor": "agent", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.TESTING0: {"phase": "testing", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.DEVELOPING1: {"phase": "developing", "actor": "agent", "reviewer": "none", "test_failures": 1, "agent_review_round": 0},
    State.TESTING1: {"phase": "testing", "actor": "pipeline", "reviewer": "none", "test_failures": 1, "agent_review_round": 0},
    State.DEVELOPING2: {"phase": "developing", "actor": "agent", "reviewer": "none", "test_failures": 2, "agent_review_round": 0},
    State.TESTING2: {"phase": "testing", "actor": "pipeline", "reviewer": "none", "test_failures": 2, "agent_review_round": 0},
    State.SECURITY_REVIEW1: {"phase": "agent_review", "actor": "agent", "reviewer": "security_agent", "test_failures": 0, "agent_review_round": 1},
    State.QUALITY_REVIEW1: {"phase": "agent_review", "actor": "agent", "reviewer": "quality_agent", "test_failures": 0, "agent_review_round": 1},
    State.REWORK_DEVELOPING: {"phase": "developing", "actor": "agent", "reviewer": "none", "test_failures": 0, "agent_review_round": 1},
    State.REWORK_TESTING: {"phase": "testing", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 1},
    State.SECURITY_REVIEW2: {"phase": "agent_review", "actor": "agent", "reviewer": "security_agent", "test_failures": 0, "agent_review_round": 2},
    State.QUALITY_REVIEW2: {"phase": "agent_review", "actor": "agent", "reviewer": "quality_agent", "test_failures": 0, "agent_review_round": 2},
    State.HUMAN_REVIEW: {"phase": "human_review", "actor": "human", "reviewer": "human", "test_failures": 0, "agent_review_round": 0},
    State.MERGED: {"phase": "merged", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.BUILDING: {"phase": "building", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.STAGING: {"phase": "staging", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.SMOKE_TESTING: {"phase": "smoke_testing", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.RELEASE_DECISION: {"phase": "release_decision", "actor": "human", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.ON_HOLD: {"phase": "on_hold", "actor": "human", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.DEPLOYING: {"phase": "deploying", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.LIVE: {"phase": "live", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
    State.ROLLED_BACK: {"phase": "rolled_back", "actor": "pipeline", "reviewer": "none", "test_failures": 0, "agent_review_round": 0},
}

#: Data variables and their initial values.
VARIABLES = {}


#: Guards: the transition is only allowed when its guard holds.
GUARDS = {
}
#: Updates of the data variables performed by transitions.
UPDATES = {
}

#: Domains of the attributes and variables.
DOMAINS = {
    "phase": ("prompting", "designing", "developing", "testing", "agent_review", "human_review", "merged", "building", "staging", "smoke_testing", "release_decision", "on_hold", "deploying", "live", "rolled_back",),
    "actor": ("human", "agent", "pipeline",),
    "reviewer": ("none", "security_agent", "quality_agent", "human",),
    "test_failures": range(0, 4),
    "agent_review_round": range(0, 4),
}

# Runtime monitors, compiled from the properties. v: attribute values of the current state,
# p: values of the past sub-formulas at the previous step, n: their values now.


def _check_0(v, p, n):
    """LTLSPEC tested_before_agent_review: G (reviewer = security_agent -> Y phase = testing)"""
    n[0] = (v["phase"] == "testing")
    return bool(((not (v["reviewer"] == "security_agent")) or p[0]))


def _check_1(v, p, n):
    """LTLSPEC security_then_quality: G (reviewer = quality_agent -> Y reviewer = security_agent)"""
    n[0] = (v["reviewer"] == "security_agent")
    return bool(((not (v["reviewer"] == "quality_agent")) or p[0]))


def _check_2(v, p, n):
    """LTLSPEC agents_before_human: G (phase = human_review -> Y reviewer = quality_agent)"""
    n[0] = (v["reviewer"] == "quality_agent")
    return bool(((not (v["phase"] == "human_review")) or p[0]))


def _check_3(v, p, n):
    """LTLSPEC human_merges: G (phase = merged -> Y (phase = human_review & actor = human))"""
    n[0] = ((v["phase"] == "human_review") and (v["actor"] == "human"))
    return bool(((not (v["phase"] == "merged")) or p[0]))


def _check_4(v, p, n):
    """INVARSPEC review_rounds_bounded: agent_review_round <= 2"""
    return bool((v["agent_review_round"] <= 2))


def _check_5(v, p, n):
    """INVARSPEC retry_budget: test_failures <= 2"""
    return bool((v["test_failures"] <= 2))


def _check_6(v, p, n):
    """LTLSPEC human_decides_release: G (phase = deploying -> Y (phase = release_decision & actor = human))"""
    n[0] = ((v["phase"] == "release_decision") and (v["actor"] == "human"))
    return bool(((not (v["phase"] == "deploying")) or p[0]))


def _check_7(v, p, n):
    """LTLSPEC deploys_the_merged_change: G (phase = deploying -> (phase != prompting S phase = merged))"""
    n[0] = ((v["phase"] == "merged")) or (((v["phase"] != "prompting")) and p[0])
    return bool(((not (v["phase"] == "deploying")) or n[0]))


def _check_8(v, p, n):
    """LTLSPEC staging_passed_for_this_release: G (phase = deploying -> (phase != prompting S phase = smoke_testing))"""
    n[0] = ((v["phase"] == "smoke_testing")) or (((v["phase"] != "prompting")) and p[0])
    return bool(((not (v["phase"] == "deploying")) or n[0]))


#: (name, kind, formula, slots, initial past values, check function).
MONITOR_SPECS = [
    ("tested_before_agent_review", "LTLSPEC", "G (reviewer = security_agent -> Y phase = testing)", 1, [False], _check_0),
    ("security_then_quality", "LTLSPEC", "G (reviewer = quality_agent -> Y reviewer = security_agent)", 1, [False], _check_1),
    ("agents_before_human", "LTLSPEC", "G (phase = human_review -> Y reviewer = quality_agent)", 1, [False], _check_2),
    ("human_merges", "LTLSPEC", "G (phase = merged -> Y (phase = human_review & actor = human))", 1, [False], _check_3),
    ("review_rounds_bounded", "INVARSPEC", "agent_review_round <= 2", 0, [], _check_4),
    ("retry_budget", "INVARSPEC", "test_failures <= 2", 0, [], _check_5),
    ("human_decides_release", "LTLSPEC", "G (phase = deploying -> Y (phase = release_decision & actor = human))", 1, [False], _check_6),
    ("deploys_the_merged_change", "LTLSPEC", "G (phase = deploying -> (phase != prompting S phase = merged))", 1, [False], _check_7),
    ("staging_passed_for_this_release", "LTLSPEC", "G (phase = deploying -> (phase != prompting S phase = smoke_testing))", 1, [False], _check_8),
]

#: The diagram itself (for the Jupyter widget, SVG display and the editor link).
DIAGRAM = {
    "name": "AgenticCodingLoop",
    "states": [
        {"name": "prompting", "label": "human prompt", "initial": True, "x": 100, "y": 80},
        {"name": "designing", "label": "agent plans, human reviews", "initial": False, "x": 360, "y": 80},
        {"name": "developing0", "label": "agent codes", "initial": False, "x": 620, "y": 80},
        {"name": "testing0", "label": "tests run", "initial": False, "x": 880, "y": 80},
        {"name": "developing1", "label": "self-heal #1", "initial": False, "x": 1140, "y": 80},
        {"name": "testing1", "label": "tests run #2", "initial": False, "x": 1400, "y": 80},
        {"name": "developing2", "label": "self-heal #2", "initial": False, "x": 1400, "y": 220},
        {"name": "testing2", "label": "tests run #3", "initial": False, "x": 1140, "y": 220},
        {"name": "security_review1", "label": "security agent review", "initial": False, "x": 880, "y": 380},
        {"name": "quality_review1", "label": "quality agent review", "initial": False, "x": 1140, "y": 380},
        {"name": "rework_developing", "label": "agent reworks review findings", "initial": False, "x": 560, "y": 520},
        {"name": "rework_testing", "label": "tests run after rework", "initial": False, "x": 860, "y": 520},
        {"name": "security_review2", "label": "security agent re-review", "initial": False, "x": 1140, "y": 520},
        {"name": "quality_review2", "label": "quality agent re-review", "initial": False, "x": 1400, "y": 520},
        {"name": "human_review", "label": "human git diff review", "initial": False, "x": 1400, "y": 380},
        {"name": "merged", "label": "PR merged", "initial": False, "x": 1400, "y": 680},
        {"name": "building", "label": "build release candidate", "initial": False, "x": 1140, "y": 680},
        {"name": "staging", "label": "deploy to staging", "initial": False, "x": 880, "y": 680},
        {"name": "smoke_testing", "label": "staging smoke tests", "initial": False, "x": 620, "y": 680},
        {"name": "release_decision", "label": "human: release?", "initial": False, "x": 360, "y": 680},
        {"name": "on_hold", "label": "release held", "initial": False, "x": 100, "y": 680},
        {"name": "deploying", "label": "deploy to production", "initial": False, "x": 360, "y": 820},
        {"name": "live", "label": "live in production", "initial": False, "x": 620, "y": 820},
        {"name": "rolled_back", "label": "rolled back", "initial": False, "x": 100, "y": 820},
    ],
    "transitions": [
        {"source": "prompting", "target": "designing", "label": "USER_SUBMIT"},
        {"source": "designing", "target": "developing0", "label": "PLAN_ACCEPTED"},
        {"source": "designing", "target": "prompting", "label": "human_clarifies"},
        {"source": "developing0", "target": "testing0", "label": "CODE_WRITTEN"},
        {"source": "testing0", "target": "security_review1", "label": "TESTS_PASSED"},
        {"source": "testing0", "target": "developing1", "label": "TESTS_FAILED"},
        {"source": "developing1", "target": "testing1", "label": "CODE_WRITTEN"},
        {"source": "testing1", "target": "security_review1", "label": "TESTS_PASSED"},
        {"source": "testing1", "target": "developing2", "label": "TESTS_FAILED"},
        {"source": "developing2", "target": "testing2", "label": "CODE_WRITTEN"},
        {"source": "testing2", "target": "security_review1", "label": "TESTS_PASSED"},
        {"source": "testing2", "target": "prompting", "label": "max_retries_reached"},
        {"source": "security_review1", "target": "quality_review1", "label": "security_approved"},
        {"source": "security_review1", "target": "rework_developing", "label": "changes_requested"},
        {"source": "quality_review1", "target": "human_review", "label": "quality_approved"},
        {"source": "quality_review1", "target": "rework_developing", "label": "changes_requested"},
        {"source": "rework_developing", "target": "rework_testing", "label": "CODE_WRITTEN"},
        {"source": "rework_testing", "target": "security_review2", "label": "TESTS_PASSED"},
        {"source": "rework_testing", "target": "prompting", "label": "TESTS_FAILED: escalate"},
        {"source": "security_review2", "target": "quality_review2", "label": "security_approved"},
        {"source": "security_review2", "target": "prompting", "label": "rejected again: escalate"},
        {"source": "quality_review2", "target": "human_review", "label": "quality_approved"},
        {"source": "quality_review2", "target": "prompting", "label": "rejected again: escalate"},
        {"source": "human_review", "target": "merged", "label": "human_approved"},
        {"source": "human_review", "target": "prompting", "label": "human_rejected"},
        {"source": "human_review", "target": "designing", "label": "redesign"},
        {"source": "merged", "target": "building", "label": "CI build"},
        {"source": "building", "target": "staging", "label": "build_ok"},
        {"source": "building", "target": "prompting", "label": "build_failed"},
        {"source": "staging", "target": "smoke_testing", "label": "deployed_to_staging"},
        {"source": "smoke_testing", "target": "release_decision", "label": "smoke_passed"},
        {"source": "smoke_testing", "target": "prompting", "label": "smoke_failed"},
        {"source": "release_decision", "target": "deploying", "label": "human_approves_release"},
        {"source": "release_decision", "target": "on_hold", "label": "human_holds"},
        {"source": "release_decision", "target": "prompting", "label": "human_rejects_release"},
        {"source": "on_hold", "target": "release_decision", "label": "human_revisits"},
        {"source": "deploying", "target": "live", "label": "health_ok"},
        {"source": "deploying", "target": "rolled_back", "label": "health_check_failed"},
        {"source": "rolled_back", "target": "release_decision", "label": "back to human"},
    ],
}


# --------------------------------------------------------------------------
# Runtime (generated, no third-party dependency; anywidget is optional)
# --------------------------------------------------------------------------


class InvalidTransition(Exception):
    """Raised when an event is not allowed in the current state of the verified model."""


class PropertyViolation(Exception):
    """Raised (in strict mode) when a runtime monitor detects a violated property."""


class Rejected:
    """Result of send() under on_invalid="return": falsy, with feedback for the caller (e.g. an LLM)."""

    def __init__(self, event, state, reason, allowed):
        self.event, self.state, self.reason, self.allowed = event, state, reason, allowed

    def __bool__(self):
        return False

    def as_feedback(self):
        """A message to hand back to an LLM so it can pick a legal step."""
        choices = ", ".join(self.allowed) if self.allowed else "none (the process is waiting)"
        return "The step %s is not allowed now (current state: %s): %s. Choose one of: %s." % (self.event, self.state, self.reason, choices)

    def __repr__(self):
        return "<Rejected %s in %s: %s>" % (self.event, self.state, self.reason)


class _Monitor:
    """Incremental monitor of G(phi) where phi only looks at the present and the past."""

    def __init__(self, name, kind, formula, slots, inits, check):
        self.name, self.kind, self.formula = name, kind, formula
        self._slots, self._inits, self._check = slots, inits, check
        self.reset()

    def reset(self):
        self._pre = list(self._inits)
        self.violated_at = None

    def step(self, v, index):
        n = [False] * self._slots
        ok = bool(self._check(v, self._pre, n))
        self._pre = n
        if not ok and self.violated_at is None:
            self.violated_at = index
        return ok


def _svg(diagram, current, visited):
    """Static SVG rendering of the diagram, current state highlighted (used by _repr_svg_)."""
    import html
    import math

    states = diagram["states"]
    pos = {}
    missing = [s for s in states if s.get("x") is None]
    for i, s in enumerate(states):
        if s.get("x") is not None:
            pos[s["name"]] = (float(s["x"]), float(s["y"]))
    for i, s in enumerate(missing):
        a = 2 * math.pi * i / max(1, len(missing))
        pos[s["name"]] = (400 + 300 * math.cos(a), 300 + 220 * math.sin(a))

    def size(s):
        text = s["name"] + ("\n" + s["label"] if s.get("label") else "")
        width = max(64, max(len(line) for line in text.split("\n")) * 7.8 + 24)
        return width / 2, 32

    radii = {s["name"]: size(s) for s in states}
    xs = [p[0] for p in pos.values()] or [0]
    ys = [p[1] for p in pos.values()] or [0]
    wmax = max(r[0] for r in radii.values()) if radii else 40
    x0, y0 = min(xs) - wmax - 40, min(ys) - 90
    x1, y1 = max(xs) + wmax + 40, max(ys) + 70
    out = [
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="%.0f %.0f %.0f %.0f" width="%.0f" '
        'font-family="Inter, system-ui, sans-serif" style="max-width:100%%;height:auto">'
        % (x0, y0, x1 - x0, y1 - y0, min(900, x1 - x0)),
        '<defs><marker id="arrow" viewBox="0 0 10 10" refX="10" refY="5" markerWidth="7" markerHeight="7" '
        'orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#7d8fa3"/></marker>'
        '<marker id="arrow-hot" viewBox="0 0 10 10" refX="10" refY="5" markerWidth="7" markerHeight="7" '
        'orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#f59f00"/></marker></defs>',
        '<rect x="%.0f" y="%.0f" width="%.0f" height="%.0f" fill="#fbfcfe"/>' % (x0, y0, x1 - x0, y1 - y0),
    ]
    pairs = {(t["source"], t["target"]) for t in diagram["transitions"]}
    last = visited[-2:] if len(visited) >= 2 else []
    for t in diagram["transitions"]:
        s, d = t["source"], t["target"]
        hot = len(last) == 2 and last[0] == s and last[1] == d
        color, marker, width = ("#f59f00", "arrow-hot", 3) if hot else ("#7d8fa3", "arrow", 1.5)
        (sx, sy), (dx, dy) = pos[s], pos[d]
        if s == d:
            rx, ry = radii[s]
            path = "M %.1f %.1f C %.1f %.1f %.1f %.1f %.1f %.1f" % (
                sx - 12, sy - ry, sx - 40, sy - ry - 55, sx + 40, sy - ry - 55, sx + 12, sy - ry)
            out.append('<path d="%s" fill="none" stroke="%s" stroke-width="%s" marker-end="url(#%s)"/>' % (path, color, width, marker))
            continue
        ang = math.atan2(dy - sy, dx - sx)
        def edge_point(cx, cy, r, a):
            rx, ry = r
            k = 1 / math.sqrt((math.cos(a) / rx) ** 2 + (math.sin(a) / ry) ** 2)
            return cx + k * math.cos(a), cy + k * math.sin(a)
        ax, ay = edge_point(sx, sy, radii[s], ang)
        bx, by = edge_point(dx, dy, radii[d], ang + math.pi)
        bend = 22 if (d, s) in pairs else 0
        mx, my = (ax + bx) / 2 - bend * math.sin(ang), (ay + by) / 2 + bend * math.cos(ang)
        out.append('<path d="M %.1f %.1f Q %.1f %.1f %.1f %.1f" fill="none" stroke="%s" stroke-width="%s" marker-end="url(#%s)"/>'
                   % (ax, ay, mx, my, bx, by, color, width, marker))
        if t.get("label"):
            out.append('<text x="%.1f" y="%.1f" font-size="10" fill="#5b6b7c" text-anchor="middle">%s</text>'
                       % (mx, my - 4, html.escape(t["label"])))
    for s in states:
        (x, y), (rx, ry) = pos[s["name"]], radii[s["name"]]
        is_current = s["name"] == current
        fill = "#f59f00" if is_current else ("#fde9c9" if s["name"] in visited else "#e8f1fb")
        stroke = "#b35c00" if is_current else ("#2f6fdf" if s.get("initial") else "#4a7fb5")
        width = 4 if is_current or s.get("initial") else 2
        out.append('<ellipse cx="%.1f" cy="%.1f" rx="%.1f" ry="%.1f" fill="%s" stroke="%s" stroke-width="%s"/>'
                   % (x, y, rx, ry, fill, stroke, width))
        lines = [s["name"]] + ([s["label"]] if s.get("label") else [])
        for i, line in enumerate(lines):
            out.append('<text x="%.1f" y="%.1f" font-size="12" font-weight="600" fill="#1b2733" '
                       'text-anchor="middle" dominant-baseline="central">%s</text>'
                       % (x, y + (i - (len(lines) - 1) / 2) * 14, html.escape(line)))
    out.append("</svg>")
    return "".join(out)


_WIDGET_ESM = r"""
import cytoscape from "https://esm.sh/cytoscape@3.34.3";

function render({ model, el }) {
  const box = document.createElement("div");
  box.style.cssText = "height:" + (model.get("height") || 420) + "px;border:1px solid #d9dfe7;border-radius:8px;background:#fbfcfe";
  const caption = document.createElement("div");
  caption.style.cssText = "font:13px system-ui,sans-serif;margin:6px 2px;color:#1b2733";
  el.append(caption, box);
  const d = model.get("diagram");
  const hasPos = d.states.every(s => s.x !== null && s.x !== undefined);
  const cy = cytoscape({
    container: box,
    elements: [
      ...d.states.map(s => ({ data: { id: s.name, label: s.label ? s.name + "\n" + s.label : s.name, initial: s.initial ? 1 : 0 },
                               position: hasPos ? { x: s.x, y: s.y } : undefined })),
      ...d.transitions.map((t, i) => ({ data: { id: "t" + i, source: t.source, target: t.target, label: t.label || "" } })),
    ],
    layout: hasPos ? { name: "preset", padding: 30 } : { name: "breadthfirst", directed: true, padding: 30 },
    style: [
      { selector: "node", style: { label: "data(label)", "text-wrap": "wrap", "text-valign": "center", "font-size": 12,
        "font-weight": 600, width: n => Math.max(64, Math.max(...n.data("label").split("\n").map(l => l.length)) * 7.8 + 24),
        height: 60, "background-color": "#e8f1fb", "border-width": 2, "border-color": "#4a7fb5", color: "#1b2733" } },
      { selector: "node[?initial]", style: { "border-width": 5, "border-style": "double", "border-color": "#2f6fdf" } },
      { selector: "node.visited", style: { "background-color": "#fde9c9" } },
      { selector: "node.current", style: { "background-color": "#f59f00", "border-color": "#b35c00", "border-width": 4 } },
      { selector: "node.next", style: { "border-color": "#2b8a3e", "border-style": "dotted", "border-width": 4 } },
      { selector: "edge", style: { "curve-style": "bezier", "target-arrow-shape": "triangle", width: 2, "line-color": "#7d8fa3",
        "target-arrow-color": "#7d8fa3", label: "data(label)", "font-size": 10, color: "#5b6b7c",
        "text-background-color": "#fbfcfe", "text-background-opacity": 0.85 } },
      { selector: "edge.last", style: { width: 4, "line-color": "#f59f00", "target-arrow-color": "#f59f00" } },
    ],
  });
  function update() {
    const state = model.get("state");
    const history = model.get("history") || [];
    cy.elements().removeClass("current visited next last");
    history.forEach(s => cy.getElementById(s).addClass("visited"));
    cy.getElementById(state).addClass("current");
    (model.get("allowed") || []).forEach(s => cy.getElementById(s).addClass("next"));
    if (history.length >= 2) {
      const a = history[history.length - 2], b = history[history.length - 1];
      cy.edges('[source = "' + a + '"][target = "' + b + '"]').addClass("last");
    }
    const v = model.get("violations") || [];
    caption.innerHTML = "<b>" + d.name + "</b> &middot; state <code>" + state + "</code> &middot; step " + (history.length - 1) +
      (model.get("last_event") ? " &middot; last event <code>" + model.get("last_event") + "</code>" : "") +
      (v.length ? ' &middot; <span style="color:#c92a2a">violated: ' + v.join(", ") + "</span>" : "");
  }
  model.on("change:state", update);
  model.on("change:history", update);
  model.on("change:violations", update);
  update();
  setTimeout(() => { cy.resize(); cy.fit(undefined, 30); }, 50);
  return () => cy.destroy();
}
export default { render };
"""


def _make_widget(fsm, height=420):
    try:
        import anywidget
        import traitlets
    except ImportError as exc:  # pragma: no cover - depends on the environment
        raise ImportError("The live diagram needs anywidget: pip install anywidget") from exc

    class StateMachineWidget(anywidget.AnyWidget):
        _esm = _WIDGET_ESM
        diagram = traitlets.Dict().tag(sync=True)
        state = traitlets.Unicode().tag(sync=True)
        history = traitlets.List().tag(sync=True)
        allowed = traitlets.List().tag(sync=True)
        last_event = traitlets.Unicode("").tag(sync=True)
        violations = traitlets.List().tag(sync=True)
        height = traitlets.Int(420).tag(sync=True)

    w = StateMachineWidget(diagram=DIAGRAM, height=height)

    def sync(fsm_, record=None):
        # The view redraws on "state", so everything it displays is set first.
        w.last_event = record["event"] if record else ""
        w.violations = [m.name for m in fsm_.monitors if m.violated_at is not None]
        w.allowed = sorted({TRANSITIONS[(fsm_.state, e)].value for e in fsm_.allowed_events()})
        w.history = [s.value for s in fsm_.visited]
        w.state = fsm_.state.value

    sync(fsm)
    fsm.subscribe(sync)
    return w


class EditorLink:
    """Streams every state change to a running nuxmv-editor (POST /api/live/<channel>).

    Open the Trace tab of the editor, choose "Live from Python" with the same channel and
    the diagram highlights the state your code is in. Network errors never interrupt the FSM.
    """

    def __init__(self, url="http://127.0.0.1:3000", channel="default", timeout=2.0):
        import re
        if not re.fullmatch(r"[\w-]{1,64}", channel):
            raise ValueError("channel: 1-64 letters, digits, _ or -")
        self.endpoint = url.rstrip("/") + "/api/live/" + channel
        self.timeout = timeout
        self.warned = False

    def __call__(self, fsm, record=None):
        import json
        import threading
        import urllib.request

        payload = {
            "diagram": DIAGRAM["name"],
            "state": fsm.state.value,
            "step": len(fsm.history),
            "event": record["event"] if record else None,
            "values": {k: v for k, v in fsm.values.items() if k != "state"},
            "allowed": [e.value for e in fsm.allowed_events()],
            "violations": [m.name for m in fsm.monitors if m.violated_at is not None],
        }

        def post():
            try:
                req = urllib.request.Request(self.endpoint, data=json.dumps(payload).encode(),
                                             headers={"content-type": "application/json"}, method="POST")
                urllib.request.urlopen(req, timeout=self.timeout).read()
            except Exception as exc:  # the FSM must not depend on the editor being up
                if not self.warned:
                    self.warned = True
                    print("EditorLink: could not reach %s (%s)" % (self.endpoint, exc))

        threading.Thread(target=post, daemon=True).start()

    def rejected(self, fsm, rejected):
        """Report a rejected event to the editor (shown in the live table)."""
        import json
        import threading
        import urllib.request

        payload = {"diagram": DIAGRAM["name"], "state": fsm.state.value, "step": len(fsm.history),
                   "rejected": {"event": rejected.event, "reason": rejected.reason},
                   "allowed": list(rejected.allowed)}

        def post():
            try:
                req = urllib.request.Request(self.endpoint, data=json.dumps(payload).encode(),
                                             headers={"content-type": "application/json"}, method="POST")
                urllib.request.urlopen(req, timeout=self.timeout).read()
            except Exception:
                pass

        threading.Thread(target=post, daemon=True).start()

    def listen(self, fsm):
        """Apply the events sent from the editor (GET /api/live/<channel>/commands, Server-Sent Events)."""
        import json
        import threading
        import time
        import urllib.request

        def run():
            while True:
                try:
                    with urllib.request.urlopen(self.endpoint + "/commands", timeout=None) as stream:
                        for raw in stream:
                            line = raw.decode("utf-8").strip()
                            if not line.startswith("data:"):
                                continue
                            command = json.loads(line[5:])
                            try:
                                fsm.send(command["event"], values=command.get("values"))
                            except Exception as exc:  # the policy decides; never kill the listener
                                print("EditorLink: command %s rejected: %s" % (command.get("event"), exc))
                except Exception:
                    time.sleep(2)  # editor not reachable yet: retry

        thread = threading.Thread(target=run, daemon=True)
        thread.start()
        return thread


class StateMachine:
    """Event-driven runtime of a verified diagram.

    * send(event) moves along a transition of the verified model or raises InvalidTransition;
      the caller (LLM, tool, human, test) chooses the event, the machine only allows legal ones.
    * allowed_events() lists the legal events, e.g. to constrain an LLM's choice.
    * on_enter_<state>(self, event, data) / on_exit_<state>(...) hooks run on each transition.
    * Runtime monitors re-check the monitorable properties on every step.
    """

    def __init__(self, initial=None, *, strict=True, listeners=(), on_invalid="raise", thread_safe=True):
        """on_invalid: what send() does with an event the model does not allow now:
        "raise" (InvalidTransition), "return" (a falsy Rejected with feedback for an LLM),
        "escalate:<EVENT>" (fire that verified escalation event instead), or a callable
        handler(fsm, rejected) whose result send() returns.
        thread_safe=False drops the lock around send() (single-threaded hosts such as Temporal workflows)."""
        start = State(initial) if initial is not None else INITIAL_STATES[0]
        if start not in INITIAL_STATES:
            raise InvalidTransition("%s is not an initial state (initial: %s)" % (start.value, ", ".join(s.value for s in INITIAL_STATES)))
        self.strict = strict
        self.state = start
        self.history = []
        self.visited = [start]
        self.free_values = {}
        self.variables = dict(VARIABLES)
        self._listeners = list(listeners)
        self._rejection_listeners = []
        self.on_invalid = on_invalid
        self.rejections = []
        self._lock = __import__("threading").RLock() if thread_safe else __import__("contextlib").nullcontext()
        self.monitors = [_Monitor(*m) for m in MONITOR_SPECS]
        self.nurv_monitors = []
        self._check_monitors(record=None)
        self._notify(None)

    @classmethod
    def restore(cls, state, variables=None, **options):
        """A machine in a given state (e.g. from a framework's serialised state). Monitors start
        afresh, so past-time properties only see the steps made from here."""
        fsm = cls(**options)
        fsm.state = State(state)
        fsm.visited = [fsm.state]
        fsm.variables.update(variables or {})
        for monitor in fsm.monitors:
            monitor.reset()
        return fsm

    # -- queries -----------------------------------------------------------
    @property
    def values(self):
        """Atoms / attribute values of the current state, as in the diagram (L(s))."""
        v = dict(LABELS[self.state])
        for name, value in self.free_values.items():
            if v.get(name) is None:
                v[name] = value
        v.update(self.variables)
        v["state"] = self.state.value
        return v

    def allowed_events(self):
        """Events with a transition from the current state whose guard holds and whose updates stay in range."""
        return [e for (s, e) in TRANSITIONS if s == self.state and self._blocked((s, e)) is None]

    def can(self, event):
        try:
            key = (self.state, _to_event(event))
        except InvalidTransition:
            return False
        return key in TRANSITIONS and self._blocked(key) is None

    def _blocked(self, key):
        """Why a transition of the table cannot fire now (None if it can)."""
        v = self.values
        guard = GUARDS.get(key)
        if guard is not None and not guard(v):
            return "its guard is false"
        update = UPDATES.get(key)
        if update is not None:
            for name, value in update(v).items():
                if value not in DOMAINS[name]:
                    return "it would set %s to %r, outside its domain" % (name, value)
        return None

    @property
    def is_terminal(self):
        return self.state in TERMINAL_STATES

    # -- transitions ---------------------------------------------------------
    def send(self, event, values=None, **data):
        """Fire an event. values: attributes left free (any) in the target state.

        Thread safe. Events the model does not allow now go to the on_invalid policy."""
        with self._lock:
            try:
                ev = _to_event(event)
            except InvalidTransition as exc:
                return self._reject(event, str(exc))
            key = (self.state, ev)
            if key not in TRANSITIONS:
                return self._reject(ev, "%s is not allowed in state %s (allowed: %s)" % (
                    ev.value, self.state.value, ", ".join(e.value for e in self.allowed_events()) or "none"))
            why = self._blocked(key)
            if why is not None:
                return self._reject(ev, "%s is not enabled in state %s: %s" % (ev.value, self.state.value, why))
            return self._fire(key, ev, values, data)

    def _reject(self, event, reason):
        rejected = Rejected(getattr(event, "value", str(event)), self.state.value, reason, [e.value for e in self.allowed_events()])
        self.rejections.append(rejected)
        for listener in list(self._rejection_listeners):
            listener(self, rejected)
        policy = self.on_invalid
        if policy == "raise":
            raise InvalidTransition(reason)
        if policy == "return":
            return rejected
        if isinstance(policy, str) and policy.startswith("escalate:"):
            escalation = policy.split(":", 1)[1]
            if self.can(escalation):
                return self.send(escalation)
            raise InvalidTransition("%s; the escalation event %s is not allowed either" % (reason, escalation))
        if callable(policy):
            return policy(self, rejected)
        raise ValueError("unknown on_invalid policy %r" % (policy,))

    def on_rejected(self, listener):
        """listener(fsm, rejected) is called for every rejected event, before the policy applies."""
        self._rejection_listeners.append(listener)
        return listener

    def _fire(self, key, event, values, data):
        source, target = self.state, TRANSITIONS[key]
        update = UPDATES.get(key)
        if update is not None:
            self.variables.update(update(self.values))
        self._set_free_values(target, values or {})
        getattr(self, "on_exit_" + source.value, _noop)(event, data)
        self.state = target
        self.visited.append(target)
        record = {"step": len(self.history) + 1, "event": event.value, "source": source.value, "target": target.value, "data": data}
        self.history.append(record)
        self._check_monitors(record)
        getattr(self, "on_enter_" + target.value, _noop)(event, data)
        self._notify(record)
        return target

    def replay(self, events):
        for e in events:
            self.send(e)
        return self

    def _set_free_values(self, target, values):
        self.free_values = {}
        for name, value in values.items():
            if name not in DOMAINS or name in VARIABLES:
                raise InvalidTransition("unknown attribute %r (variables change only through transitions)" % (name,))
            if LABELS[target].get(name) is not None:
                raise InvalidTransition("%s is fixed to %r in state %s" % (name, LABELS[target][name], target.value))
            if value not in DOMAINS[name]:
                raise InvalidTransition("%r is not in the domain of %s" % (value, name))
            self.free_values[name] = value

    # -- monitors and listeners ---------------------------------------------
    def _check_monitors(self, record):
        v = self.values
        broken = [m for m in self.monitors if not m.step(v, len(self.history))]
        if broken and self.strict:
            names = ", ".join("%s %s (%s)" % (m.kind, m.name, m.formula) for m in broken)
            raise PropertyViolation("step %d, state %s: %s" % (len(self.history), self.state.value, names))

    def subscribe(self, listener):
        """listener(fsm, record) is called after every transition (record is None initially)."""
        self._listeners.append(listener)
        return listener

    def _notify(self, record):
        for listener in list(self._listeners):
            listener(self, record)

    # -- display --------------------------------------------------------------
    def widget(self, height=420):
        """Live diagram for Jupyter (needs anywidget); follows every transition."""
        return _make_widget(self, height)

    def record_to(self, path):
        """Append every step to a JSON Lines file, for trace conformance checking in the editor (Trace -> Check a recorded run) or with "nxd conform"."""
        import json
        import time

        def write(fsm, record=None):
            line = {
                "step": len(fsm.history),
                "event": record["event"] if record else None,
                "source": record["source"] if record else None,
                "state": fsm.state.value,
                "values": {k: v for k, v in fsm.values.items() if k != "state"},
                "time": time.time(),
            }
            with open(path, "a", encoding="utf-8") as out:
                out.write(json.dumps(line) + "\n")

        self.subscribe(write)
        write(self, None)
        return write

    def add_nurv_monitor(self, module):
        """Attach a full-LTL monitor generated by NuRV (nuxmv-editor: Python tab -> NuRV monitors,
        or "nxd nurv"). Its verdict is "unknown" until the observed run decides the property for
        every continuation the model allows, then "true" or "false"; "false" raises
        PropertyViolation in strict mode. Returns a dict holding the current verdict."""
        monitor = module.Monitor()
        status = {"name": module.__name__, "verdict": "unknown", "decided_at": None}
        reset = [module.RV_reset.HARD_RESET]

        def encode(values):
            # NuRV encodes enumeration symbols as module-level integer constants.
            out = {}
            for key, value in values.items():
                if isinstance(value, str) and isinstance(getattr(module, value, None), int):
                    out[key] = getattr(module, value)
                elif isinstance(value, bool):
                    out[key] = int(value)
                else:
                    out[key] = value
            return out

        def step(fsm, record=None):
            verdict = monitor.run(encode(fsm.values), reset[0])
            reset[0] = module.RV_reset.NO_RESET
            name = verdict.name.replace("RV_", "").lower()
            if name in ("true", "false") and status["decided_at"] is None:
                status["decided_at"] = len(fsm.history)
            status["verdict"] = name
            if name == "false" and fsm.strict:
                raise PropertyViolation("step %d, state %s: NuRV monitor %s is false" % (len(fsm.history), fsm.state.value, status["name"]))

        self.subscribe(step)
        step(self)
        self.nurv_monitors.append(status)
        return status

    def link_editor(self, url="http://127.0.0.1:3000", channel="default", commands=False):
        """Stream state changes to a running nuxmv-editor. With commands=True the editor can also
        send events back (e.g. a human approving a step); they go through send(), so only verified
        transitions can happen."""
        link = EditorLink(url, channel)
        self.subscribe(link)
        self.on_rejected(link.rejected)
        link(self, None)
        if commands:
            link.listen(self)
        return link

    def enable_tracing(self, tracer=None):
        """Emit an OpenTelemetry span per transition ("fsm.transition") and per rejected event
        ("fsm.rejected"), with fsm.* attributes; recorded spans can be checked against the model
        in the editor (Trace -> Check a recorded run). Needs opentelemetry-api."""
        try:
            from opentelemetry import trace
            from opentelemetry.trace import Status, StatusCode
        except ImportError as exc:  # pragma: no cover - depends on the environment
            raise ImportError("Tracing needs OpenTelemetry: pip install opentelemetry-api opentelemetry-sdk") from exc
        tracer = tracer or trace.get_tracer("nuxmv-editor-js." + DIAGRAM["name"])

        def attributes(fsm, record=None):
            a = {"fsm.machine": DIAGRAM["name"], "fsm.state": fsm.state.value, "fsm.step": len(fsm.history)}
            if record:
                a["fsm.event"] = record["event"]
                a["fsm.source"] = record["source"]
            for k, v in fsm.values.items():
                if k != "state" and isinstance(v, (bool, int, float, str)):
                    a["fsm.value." + k] = v
            violated = [m.name for m in fsm.monitors if m.violated_at is not None]
            if violated:
                a["fsm.violations"] = violated
            return a

        def on_step(fsm, record=None):
            with tracer.start_as_current_span("fsm.transition" if record else "fsm.start", attributes=attributes(fsm, record)):
                pass

        def on_rejected(fsm, rejected):
            a = attributes(fsm)
            a.update({"fsm.rejected": True, "fsm.event": rejected.event, "fsm.reason": rejected.reason})
            with tracer.start_as_current_span("fsm.rejected", attributes=a) as span:
                span.set_status(Status(StatusCode.ERROR, rejected.reason))

        self.subscribe(on_step)
        self.on_rejected(on_rejected)
        on_step(self, None)
        return tracer

    def _repr_svg_(self):
        return _svg(DIAGRAM, self.state.value, [s.value for s in self.visited])

    def __repr__(self):
        return "<%s state=%s step=%d>" % (type(self).__name__, self.state.value, len(self.history))


def _noop(*_args, **_kwargs):
    return None


def _to_event(event):
    """Accepts an Event, its name, or the transition label as written in the diagram."""
    import re
    if isinstance(event, Event):
        return event
    try:
        return Event(event)
    except ValueError:
        pass
    words = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", str(event))
    name = "_".join(w for w in re.split(r"[^A-Za-z0-9]+", words) if w).upper()
    try:
        return Event(name)
    except ValueError:
        raise InvalidTransition("unknown event %r" % (event,)) from None


class AgenticCodingLoopFSM(StateMachine):
    """AgenticCodingLoop. Subclass it and define on_enter_<state>(self, event, data) hooks to attach work to states."""


__all__ = [
    "AgenticCodingLoopFSM", "State", "Event", "TRANSITIONS", "LABELS", "INITIAL_STATES", "TERMINAL_STATES",
    "InvalidTransition", "PropertyViolation", "Rejected", "EditorLink",
]


In [ ]:
import importlib, time
import agentic_coding_loop_fsm as m
importlib.reload(m)

fsm = m.AgenticCodingLoopFSM()
fsm  # rendered as a diagram with the current state highlighted

## Live diagram

The widget below follows the state of `fsm`: run the next cells and watch it move (current state in orange, possible next states dotted green).

In [ ]:
try:
    live = fsm.widget(height=460)
except ImportError as error:
    live = None
    print(error)
live

## Walking through the model

A shortest path from the initial state to the final state `live`. Each `send` is checked against the verified transition table.

In [ ]:
path = [
    "USER_SUBMIT",
    "PLAN_ACCEPTED",
    "CODE_WRITTEN",
    "TESTS_PASSED",
    "SECURITY_APPROVED",
    "QUALITY_APPROVED",
    "HUMAN_APPROVED",
    "CI_BUILD",
    "BUILD_OK",
    "DEPLOYED_TO_STAGING",
    "SMOKE_PASSED",
    "HUMAN_APPROVES_RELEASE",
    "HEALTH_OK",
]

for event in path:
    fsm.send(event)
    print(f"{event:>30}  ->  {fsm.state.value}")
    time.sleep(0.4)  # let the live diagram animate

In [ ]:
fsm  # the diagram again, with the visited states

## The verified model is enforced

An event that has no transition from the current state is rejected, whatever produced it: an LLM, a tool or a person.

In [ ]:
fresh = m.AgenticCodingLoopFSM()
print("allowed:", [e.value for e in fresh.allowed_events()])
try:
    fresh.send("PLAN_ACCEPTED")
except m.InvalidTransition as error:
    print("rejected:", error)

## Constraining an LLM to legal moves

Instead of letting the model decide freely, offer it only the events allowed in the current state, for example as the `enum` of a tool parameter. Whatever it answers, `send` still validates it.

In [ ]:
def next_step_tool(fsm):
    """Tool definition whose only valid arguments are the legal next events."""
    return {
        "name": "next_step",
        "description": f"Choose the next step. Current state: {fsm.state.value}.",
        "input_schema": {
            "type": "object",
            "properties": {"event": {"type": "string", "enum": [e.value for e in fsm.allowed_events()]}},
            "required": ["event"],
        },
    }

next_step_tool(m.AgenticCodingLoopFSM())

## Attaching work to states

Subclass the machine and define `on_enter_<state>` (or `on_exit_<state>`) hooks: this is where LLM calls, tools and human approvals go. The control flow itself stays the verified one.

In [ ]:
class MyAgenticCodingLoopFSM(m.AgenticCodingLoopFSM):
    def on_enter_designing(self, event, data):
        print(f"entered designing via {event.value}; call your agent or tool here")

agent = MyAgenticCodingLoopFSM()
agent.send("USER_SUBMIT")

## Runtime monitors

Monitored properties are re-checked after every transition; in strict mode (the default) a violation raises `PropertyViolation`.

In [ ]:
for monitor in fsm.monitors:
    status = "ok" if monitor.violated_at is None else f"violated at step {monitor.violated_at}"
    print(f"{monitor.kind:10} {monitor.name:30} {status}")

## Live state in nuxmv-editor

The same states can be shown in the editor while any Python process runs: open the **Trace** tab, choose **Live from Python**, pick a channel, then link the machine to it.

In [ ]:
# fsm.link_editor("http://localhost:3000", channel="notebook")
# fsm.send(...)  # every transition now also moves the editor's diagram